# GISLR-GapCorpus v1 — Kaggle build (self-contained)

**Generated file — do not edit.** Regenerate with
`python -m sb.recognize.sequences.gapcorpus_kaggle_notebook` (signbridge commit
`ffb4563`, embedded-code sha256 `d7c85f1632e0`). Built for TODO §17: training a
dedicated sign-vs-gap detector — decoupled from any classifier — that both the
deployed continuous GRU/LSTM (§12.3) and the record-then-recognize BiLSTM hybrid
(§16.2) can use as their segmentation source instead of a fixed heuristic.

**How to run on Kaggle**
1. *New Notebook* → *File → Import Notebook* → this `.ipynb`.
2. *Add Input* → your dataset `bracu23101281/gislr-stratified` (found automatically
   under `/kaggle/input`; set `INPUT_DIR` in the setup cell if not).
3. Accelerator: **None** (CPU only; no GPU needed). Internet: not needed.
4. *Save Version* → **Save & Run All (Commit)**.
5. When it finishes: Output tab → **New Dataset**, e.g. `GISLR-GapCorpus` (private by default).

**Output limit.** Kaggle keeps at most ~20 GB of `/kaggle/working`. `OUTPUT_LIMIT_GB`
stops the build early (with instructions) if the projected size would exceed it;
`STORAGE_DTYPE = "float16"` halves it.

**What it builds**, from GISLR_Stratified's **`train.csv` + `test.csv` combined**
(4 independent random passes over every clip):
- Random-order, random-length (2–6 signs)
  sequences, one signer each — no sentence structure, no coverage requirement.
- Rest: **lowered** (realistic — arms ramp to hip height, hands vanish only once out
  of frame) with probability 0.7, else hands-absent
  (GISLR-Sentences v1's style) — a deliberate mix so a model trained on this handles
  both.
- Transitions: 0–15 interpolated frames between signs.
- Per sequence: `landmarks (T,543,3)`, `frame_kind` (0 sign / 1 transition / 2 rest),
  **`gap` (T,) uint8** — 1 on every non-sign frame, the label this dataset exists for
  — `segments`, `labels`, `lowered_rest` (bool).
- **`train.csv` / `test.csv`**: one row per sequence — `glosses`, `starts`/`ends`
  (the frame each sign starts/ends — everything outside those spans is a gap),
  `npz_relpath`, `pass_id`, `lowered_rest`, `participant_id`. Split by **participant**
  (85% of signers in train), independent of
  GISLR_Stratified's own clip-level split — this model's generalization question is
  new signers, not new glosses.

## Setup

In [ ]:
# ============================================================
# Tunables, config, paths
# ============================================================
import hashlib
import json
import os
import shutil
import sys
import types
from concurrent.futures import ThreadPoolExecutor
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from tqdm.auto import tqdm

INPUT_DIR = None             # None = auto-detect GISLR_Stratified under /kaggle/input
OUT_DIR = Path("/kaggle/working/gislr-gapcorpus")
STORAGE_DTYPE = "float32"    # "float16" halves the size
OUTPUT_LIMIT_GB = 19.0
N_WORKERS = os.cpu_count() or 4

CONFIG = json.loads(r'''{
  "dataset_version": "v1",
  "seed": 20260929,
  "signs_per_seq": [
    2,
    6
  ],
  "n_passes": 4,
  "gap_frames": [
    0,
    15
  ],
  "rest_frames": [
    5,
    25
  ],
  "rest_ramp_frames": 6,
  "hand_visible_max": 0.95,
  "rest_default_drop": 0.25,
  "rest_jitter": 0.01,
  "p_lowered_rest": 0.7,
  "train_signer_frac": 0.85,
  "n_roundtrip_checks": 300
}''')
CONFIG_SHA = hashlib.sha256(json.dumps(CONFIG, sort_keys=True).encode()).hexdigest()
CODE_SHA256 = "d7c85f1632e036a99a3e85c7f649e41a32fd800c54aa9b8a6287ed995260cfb0"
SIGNBRIDGE_COMMIT = "ffb4563"


def find_input() -> Path:
    for depth in range(1, 6):
        hits = [p.parent for p in Path("/kaggle/input").glob("/".join(["*"] * depth) + "/test.csv")
                if (p.parent / "train.csv").is_file() and (p.parent / "test").is_dir()]
        if hits:
            assert len(hits) == 1, f"several GISLR_Stratified candidates, set INPUT_DIR: {hits}"
            return hits[0]
    raise FileNotFoundError("GISLR_Stratified not found under /kaggle/input -- add it as an input")


DATA_DIR = Path(INPUT_DIR) if INPUT_DIR else find_input()
SEQ_DIR = OUT_DIR / "sequences"
ASSETS_DIR = OUT_DIR / "assets"
PLAN_PATH = OUT_DIR / "plan.json"
MANIFEST_PATH = OUT_DIR / "manifest.json"
for d in (SEQ_DIR, ASSETS_DIR):
    d.mkdir(parents=True, exist_ok=True)


def write_json(path: Path, obj) -> None:
    tmp = path.with_suffix(".tmp")
    tmp.write_text(json.dumps(obj, indent=1), encoding="utf-8")
    os.replace(tmp, path)


print(f"input:   {DATA_DIR}")
print(f"output:  {OUT_DIR}  ({STORAGE_DTYPE}, {N_WORKERS} workers)")
print(f"config:  sha256={CONFIG_SHA[:12]}   code sha256={CODE_SHA256[:12]} (commit {SIGNBRIDGE_COMMIT})")

## Embedded signbridge code

The exact source of the modules the build uses, registered under their real names,
so `compose`'s own imports resolve unchanged. Numpy/stdlib-only.

In [ ]:
# ============================================================
# Register the embedded modules
# ============================================================
_SOURCES = {}
_SOURCES['sb.core.schema'] = r'''"""The landmark-tensor contract between extraction (stage 1) and training (stage 2).

This is the interface both datasets and both pipeline stages agree on, and until
now it was a paragraph of README prose plus the same 543-row layout restated in
``extraction.GROUP_LAYOUT``, ``quality.GROUPS`` and ``subsets``' docstring. It is
load-bearing for a cross-dataset claim — POPSIGN's extracted npz is written in
*GISLR holistic row order* precisely so that ``subsets.py`` indices apply to it
unchanged — and nothing checked it (TODO §9.4).

Row layout (543 rows/frame, xyz each)::

    rows   0-467  face      (468 face-mesh landmarks; row == face-mesh index)
    rows 468-488  left_hand (21)
    rows 489-521  pose      (33; holistic row = 489 + pose index)
    rows 522-542  right_hand(21)

On-disk unit (one video, POPSIGN extraction)::

    <root>/data/raw/popsign/<split>/<label>/<video_id>.npz
        landmarks   (T, 543, 3) float16, NaN where undetected
        fps         float32 scalar
        num_frames  int32 scalar   == landmarks.shape[0]

The label is carried by the path, not stored inside the file.

**NaN is data, not corruption.** An undetected group is NaN for that frame, and
that is what ``quality.py``'s detection-rate proxies measure and what the
1st-place feature pipeline normalizes over. A validator that rejected NaN would
reject every real file.

Cheap by design: :func:`validate_tensor` does O(1) structural checks by default,
so it can sit on the extraction write path for 30k+ videos (measured: 0.3 us a
call). The value scan (infinities) is opt-in and used by the npz reader, which
is a diagnostic path.
"""

from dataclasses import dataclass
from pathlib import Path

import numpy as np

SPEC_VERSION = 1

N_LANDMARKS = 543  # holistic rows per frame
N_COORDS = 3  # x, y, z
STORAGE_DTYPE = np.float16  # what extraction writes
MISSING_VALUE = "nan"  # how an undetected landmark is represented


@dataclass(frozen=True)
class LandmarkGroup:
    """One contiguous block of holistic rows."""

    name: str
    offset: int
    size: int
    result_attr: str  # the MediaPipe HolisticLandmarkerResult attribute

    @property
    def slice(self) -> slice:
        return slice(self.offset, self.offset + self.size)


GROUPS: tuple[LandmarkGroup, ...] = (
    LandmarkGroup("face", 0, 468, "face_landmarks"),
    LandmarkGroup("left_hand", 468, 21, "left_hand_landmarks"),
    LandmarkGroup("pose", 489, 33, "pose_landmarks"),
    LandmarkGroup("right_hand", 522, 21, "right_hand_landmarks"),
)

GROUP_SLICES: dict[str, slice] = {g.name: g.slice for g in GROUPS}
POSE_OFFSET: int = GROUP_SLICES["pose"].start  # holistic row of pose landmark 0

# npz keys every extracted file carries, and the dtype each must have
NPZ_KEYS: dict[str, type] = {
    "landmarks": np.float16,
    "fps": np.float32,
    "num_frames": np.int32,
}

assert sum(g.size for g in GROUPS) == N_LANDMARKS, "group sizes must tile the tensor"
assert [g.offset for g in GROUPS] == list(
    np.cumsum([0, *(g.size for g in GROUPS[:-1])])
), "groups must be contiguous and in order"


class SpecError(ValueError):
    """A landmark tensor or npz violates LANDMARK_TENSOR_V1."""


def spec() -> dict:
    """The contract as data — for embedding in a manifest or a run record."""
    return {
        "spec": "LANDMARK_TENSOR",
        "version": SPEC_VERSION,
        "n_landmarks": N_LANDMARKS,
        "n_coords": N_COORDS,
        "storage_dtype": np.dtype(STORAGE_DTYPE).name,
        "missing_value": MISSING_VALUE,
        "row_order": "gislr-holistic",
        "groups": [
            {"name": g.name, "offset": g.offset, "size": g.size} for g in GROUPS
        ],
        "npz_keys": {k: np.dtype(v).name for k, v in NPZ_KEYS.items()},
    }


def validate_tensor(
    arr: np.ndarray,
    *,
    dtype: type | None = STORAGE_DTYPE,
    check_values: bool = False,
    where: str = "landmarks",
) -> np.ndarray:
    """Structural check of a ``(T, 543, 3)`` landmark tensor; returns it.

    ``dtype=None`` accepts any floating type — loaders legitimately widen
    float16 to float32. ``check_values=True`` additionally scans for infinities;
    it is O(size), so the extraction write path leaves it off.

    T == 0 is valid: a video whose frames all failed to decode is a real,
    recorded outcome, not a malformed file.
    """
    if not isinstance(arr, np.ndarray):
        raise SpecError(f"{where}: expected ndarray, got {type(arr).__name__}")
    if arr.ndim != 3:
        raise SpecError(f"{where}: expected (T, {N_LANDMARKS}, {N_COORDS}), got {arr.shape}")
    if arr.shape[1] != N_LANDMARKS or arr.shape[2] != N_COORDS:
        raise SpecError(
            f"{where}: expected (T, {N_LANDMARKS}, {N_COORDS}), got {arr.shape} — "
            "row order is GISLR holistic (face 0-467, left_hand 468-488, "
            "pose 489-521, right_hand 522-542); a different row count means the "
            "subsets.py indices no longer apply to this file"
        )
    if dtype is not None and arr.dtype != np.dtype(dtype):
        raise SpecError(f"{where}: expected dtype {np.dtype(dtype).name}, got {arr.dtype}")
    if dtype is None and arr.dtype.kind != "f":
        raise SpecError(f"{where}: expected a floating dtype, got {arr.dtype}")
    if check_values and arr.size and np.isinf(arr).any():
        # inf cannot come out of the pipeline and silently destroys any
        # normalization downstream. An all-NaN tensor, by contrast, is a real
        # extraction outcome (nothing detected in that clip) and is a QUALITY
        # signal for quality.py to score, never a spec violation.
        raise SpecError(f"{where}: contains infinities (undetected must be NaN)")
    return arr


def validate_npz(path: Path | str, *, check_values: bool = True) -> dict:
    """Validate one extracted npz against the spec; returns a small summary.

    Reads `landmarks` fully, so this is a diagnostic/verification path rather
    than something to call per-sample in a training loop.
    """
    path = Path(path)
    with np.load(path) as d:
        missing = [k for k in NPZ_KEYS if k not in d]
        if missing:
            raise SpecError(f"{path.name}: missing npz key(s) {missing}")
        arr = d["landmarks"]
        fps = d["fps"]
        num_frames = d["num_frames"]
        validate_tensor(arr, check_values=check_values, where=path.name)
        for key, want in (("fps", np.float32), ("num_frames", np.int32)):
            got = {"fps": fps, "num_frames": num_frames}[key]
            if np.dtype(got.dtype) != np.dtype(want):
                raise SpecError(
                    f"{path.name}: {key} dtype {got.dtype}, expected {np.dtype(want).name}")
        if int(num_frames) != arr.shape[0]:
            raise SpecError(
                f"{path.name}: num_frames={int(num_frames)} but landmarks hold "
                f"{arr.shape[0]} frames")
        detected = {
            g.name: float(1.0 - np.isnan(arr[:, g.slice, 0]).all(axis=1).mean())
            if arr.shape[0] else 0.0
            for g in GROUPS
        }
    return {
        "path": str(path),
        "n_frames": int(num_frames),
        "fps": float(fps),
        "dtype": str(arr.dtype),
        "detection_rate": detected,
    }
'''
_SOURCES['sb.recognize.features.gislr_stratified'] = r'''"""On-disk read for the GISLR_Stratified npz dataset — shared by both feature
pipelines (``base_v1`` zeroes NaN, ``firstplace_v1`` preserves it, ``evaluate``
reads it directly), so the layout fix below lives in one place.

**Row-order mismatch.** The producing notebook wrote ``landmarks`` as
``pose[0:33], face[33:501], left_hand[501:522], right_hand[522:543]`` — NOT
this repo's canonical gislr-holistic order (``sb.core.schema.GROUPS``:
``face[0:468], left_hand[468:489], pose[489:522], right_hand[522:543]``) that
every ``sb.core.subsets`` index array (ME_126, FP_118, ...) is defined
against. Reading the file as-is would silently hand a subset the wrong
physical landmarks. ``CANONICAL_TO_STORED`` is the fixed permutation that
undoes this at read time, so callers keep indexing with canonical subset
arrays and get back the landmarks those indices actually name.
"""

from pathlib import Path

import numpy as np

from sb.core.schema import N_LANDMARKS, validate_tensor

_COORD_INDEX = {"x": 0, "y": 1, "z": 2}

CANONICAL_TO_STORED = np.empty(N_LANDMARKS, dtype=np.int64)
CANONICAL_TO_STORED[0:468] = np.arange(33, 501)  # face
CANONICAL_TO_STORED[468:489] = np.arange(501, 522)  # left_hand
CANONICAL_TO_STORED[489:522] = np.arange(0, 33)  # pose
CANONICAL_TO_STORED[522:543] = np.arange(522, 543)  # right_hand (already aligned)


def load_npz(path: Path | str, rows: np.ndarray, coords: str) -> np.ndarray:
    """One GISLR_Stratified npz -> ``(T, len(rows), len(coords))`` float32.

    ``rows`` are canonical gislr-holistic indices (a subset's ``.array``);
    they're remapped onto the npz's own on-disk order before selecting. NaN is
    preserved — callers decide whether to zero it (``base_v1`` does,
    ``firstplace_v1`` must not).
    """
    path = Path(path)
    with np.load(path) as d:
        arr = d["landmarks"]
    validate_tensor(arr, dtype=None, where=path.name)
    mapped_rows = CANONICAL_TO_STORED[np.asarray(rows)]
    coord_idx = [_COORD_INDEX[c] for c in coords]
    return arr[:, mapped_rows, :][:, :, coord_idx].astype(np.float32)
'''
_SOURCES['sb.recognize.sequences.corpus'] = r'''"""The sentence corpus: ASL-gloss-order sentences over the 250 GISLR glosses.

Committed and versioned beside this module (``corpus_data/``), like
``sb.rescore``'s prompts: a dataset built from ``sentences.v1`` records
:func:`corpus_sha256` so an edited corpus can never masquerade as the one a
dataset was built from. A new wording is a new version directory, never an
in-place edit of a published one.

Layout of one version::

    corpus_data/lexicon.v1.json         {gloss: [part-of-speech tags]}
    corpus_data/sentences.v1/<theme>.txt  one sentence per line, glosses separated
                                     by spaces, '#' starts a comment

Sentence ids are ``<theme>-<line index within theme>`` so adding a line to one
theme file never renumbers another theme's sentences.
"""

from __future__ import annotations

import hashlib
import json
from collections import Counter
from dataclasses import dataclass
from pathlib import Path

CORPUS_DIR = Path(__file__).parent / "corpus_data"

POS_TAGS = (
    "noun", "verb", "adj", "time", "wh", "pronoun", "neg", "modal",
    "prep", "conj", "quant", "greeting", "response",
)


@dataclass(frozen=True)
class Sentence:
    id: str
    theme: str
    glosses: tuple[str, ...]

    def __len__(self) -> int:
        return len(self.glosses)


def lexicon_path(version: str = "v1") -> Path:
    return CORPUS_DIR / f"lexicon.{version}.json"


def sentences_dir(version: str = "v1") -> Path:
    return CORPUS_DIR / f"sentences.{version}"


def load_lexicon(version: str = "v1") -> dict[str, list[str]]:
    """``{gloss: [pos, ...]}`` -- a gloss can carry several tags (``drink``
    is noun and verb)."""
    lex = json.loads(lexicon_path(version).read_text(encoding="utf-8"))
    bad = {g: t for g, t in lex.items() if set(t) - set(POS_TAGS)}
    if bad:
        raise ValueError(f"unknown POS tags in lexicon.{version}: {bad}")
    return lex


def load_sentences(version: str = "v1") -> list[Sentence]:
    """Every sentence of one corpus version, themes in sorted-filename order."""
    out = []
    for path in sorted(sentences_dir(version).glob("*.txt")):
        theme = path.stem
        i = 0
        for line in path.read_text(encoding="utf-8").splitlines():
            line = line.split("#", 1)[0].strip()
            if not line:
                continue
            out.append(Sentence(f"{theme}-{i:03d}", theme, tuple(line.split())))
            i += 1
    return out


def corpus_sha256(version: str = "v1") -> str:
    """Content hash over the lexicon and every sentence file (name + bytes),
    recorded by anything built from the corpus. Line endings are normalized
    first: git's autocrlf rewrites them per checkout, and the same corpus
    must hash the same on every machine."""
    h = hashlib.sha256()
    files = [lexicon_path(version), *sorted(sentences_dir(version).glob("*.txt"))]
    for path in files:
        h.update(path.name.encode())
        h.update(path.read_bytes().replace(b"\r\n", b"\n"))
    return h.hexdigest()


def validate(
    sentences: list[Sentence],
    vocab,
    *,
    min_len: int = 2,
    max_len: int = 8,
    min_per_gloss: int = 12,
) -> dict:
    """Hard failures raise; soft coverage gaps are returned for the notebook
    to print. Hard: a token outside the vocabulary, a sentence outside
    ``[min_len, max_len]``, or an exact duplicate sentence (it would silently
    double that sentence's sampling weight). Soft: glosses appearing in fewer
    than ``min_per_gloss`` sentences -- the composer can still use every clip
    of such a gloss, but only by repeating its few sentences."""
    vocab = set(vocab)
    unknown = [(s.id, g) for s in sentences for g in s.glosses if g not in vocab]
    if unknown:
        raise ValueError(f"{len(unknown)} tokens outside the vocabulary, e.g. {unknown[:10]}")
    bad_len = [s.id for s in sentences if not min_len <= len(s) <= max_len]
    if bad_len:
        raise ValueError(f"{len(bad_len)} sentences outside [{min_len}, {max_len}]: {bad_len[:10]}")
    seen: dict[tuple[str, ...], str] = {}
    dups = []
    for s in sentences:
        if s.glosses in seen:
            dups.append((seen[s.glosses], s.id))
        seen.setdefault(s.glosses, s.id)
    if dups:
        raise ValueError(f"{len(dups)} duplicate sentences: {dups[:10]}")

    # coverage = number of SENTENCES containing each gloss (not token count)
    coverage = Counter(g for s in sentences for g in set(s.glosses))
    missing = sorted(vocab - set(coverage))
    thin = sorted((g, coverage[g]) for g in vocab if 0 < coverage[g] < min_per_gloss)
    lengths = Counter(len(s) for s in sentences)
    return {
        "n_sentences": len(sentences),
        "n_themes": len({s.theme for s in sentences}),
        "coverage": dict(coverage),
        "missing": missing,
        "thin": thin,
        "length_hist": dict(sorted(lengths.items())),
        "mean_len": sum(len(s) for s in sentences) / max(len(sentences), 1),
    }
'''
_SOURCES['sb.recognize.sequences.compose'] = r'''"""Sentences -> continuous landmark sequences with per-frame labels.

Two phases, deliberately split so the expensive one is resumable and the
cheap one is reviewable before anything is written:

1. **Plan** (:func:`plan_sentences`, :func:`plan_control`) -- pure
   bookkeeping over ``test.csv``: which clip fills which slot of which
   sequence, plus every synthesized gap/rest length. A plan row fully
   determines its sequence, so materialization is deterministic and can be
   interrupted and resumed per sequence.
2. **Materialize** (:func:`materialize`) -- read the clips, synthesize the
   non-sign frames, return the arrays for one sequence.

**Clip usage.** ``plan_sentences`` is coverage-driven: it keeps drawing
sentences until every source clip has been placed at least once, preferring
sentences whose glosses the chosen signer still has *unused* clips for. A
slot is only filled with an already-used clip when no sentence can be
completed from unused ones -- the ``reused`` flag records each such slot so
the reuse rate is measurable, never hidden. ``plan_control`` uses every clip
exactly once, in random order (no sentence structure): the "no language
prior" control.

**One signer per sequence.** Every slot of a sequence comes from the same
``participant_id`` -- a real continuous signer does not change identity (and
body position) at every sign boundary. A sentence is only eligible for a
signer who recorded every gloss in it.

**Synthesized frames** (GISLR clips are trimmed to the sign; nothing in the
dataset shows a hand at rest or moving between signs):

- *transition* between consecutive signs: ``g`` frames linearly interpolated,
  per landmark and coordinate, from the last frame of one clip to the first
  frame of the next. NaN on either end stays NaN (a hand absent at either
  end is not invented).
- *rest* before the first and after the last sign: the adjacent clip's edge
  frame held still with small Gaussian jitter, **both hands NaN** -- what
  MediaPipe emits while the hands are down and out of frame. The body and
  face are present.

Both are labeled null (``NULL_LABEL``) and tagged in ``frame_kind`` so an
evaluation can drop them (``frame_kind == SIGN`` recovers the hard-cut,
back-to-back concatenation exactly) or score them separately.

Landmarks are written in the repo's canonical gislr-holistic row order
(``sb.core.schema.GROUPS``), NOT the source files' stored order -- the
permutation is undone once here, so readers of this dataset never need
``CANONICAL_TO_STORED``.
"""

from __future__ import annotations

import os
from pathlib import Path

import numpy as np
import pandas as pd

from sb.core.schema import GROUPS, N_LANDMARKS, validate_tensor
from sb.recognize.features.gislr_stratified import load_npz
from sb.recognize.sequences.corpus import Sentence

NULL_LABEL = -1
SIGN, TRANSITION, REST = 0, 1, 2  # frame_kind values
FRAME_KINDS = {SIGN: "sign", TRANSITION: "transition", REST: "rest"}

_ALL_ROWS = np.arange(N_LANDMARKS)
HAND_ROWS = np.concatenate([
    np.arange(g.offset, g.offset + g.size) for g in GROUPS if g.name in ("left_hand", "right_hand")
])


# ---------------------------------------------------------------------------
# plan
# ---------------------------------------------------------------------------

def _synth_lengths(n_signs: int, rng: np.random.Generator, gap_range, rest_range) -> dict:
    lo, hi = gap_range
    rlo, rhi = rest_range
    return {
        "gaps": [int(g) for g in rng.integers(lo, hi + 1, size=max(n_signs - 1, 0))],
        "rest_in": int(rng.integers(rlo, rhi + 1)),
        "rest_out": int(rng.integers(rlo, rhi + 1)),
    }


def plan_sentences(
    df: pd.DataFrame,
    sentences: list[Sentence],
    rng: np.random.Generator,
    *,
    gap_range: tuple[int, int],
    rest_range: tuple[int, int],
    reuse_penalty: float = 2.0,
    repeat_penalty: float = 0.15,
    length_bonus: float = 0.0,
    pressure_weight: float = 0.0,
    progress=None,
) -> tuple[list[dict], dict]:
    """Coverage-driven sentence instantiation over every clip in ``df``.

    Loop: take the signer with the most unused clips left; score every
    sentence eligible for that signer as

        fresh_frac - reuse_penalty * reuse_frac - repeat_penalty * times_used_by_signer
        + length_bonus * n_fresh / max_len + pressure_weight * pressure

    where ``fresh_frac`` is the fraction of the sentence's slots fillable with
    that signer's unused clips and ``pressure`` is the mean number of unused
    clips its glosses still have, relative to the signer's average; take the
    best (random tiebreak), fill it. Stops when every clip is placed. Returns
    ``(plan, stats)``.

    Scoring by *fraction* keeps any fully-fresh sentence ahead of a partly
    reused one; ``length_bonus`` then breaks the tie toward longer sentences
    (without it, 2-gloss sentences win because they are the easiest to fill
    fresh). ``pressure`` drains the glosses a signer has most of first, so the
    last clips left are not a scatter no sentence can use without reuse.
    ``repeat_penalty`` spreads a signer across many sentences instead of
    repeating a favourite.
    """
    glosses = sorted(df["sign"].unique())
    gidx = {g: i for i, g in enumerate(glosses)}
    G, N = len(glosses), len(sentences)
    S = np.zeros((N, G), dtype=np.int32)
    for i, s in enumerate(sentences):
        for g in s.glosses:
            S[i, gidx[g]] += 1
    lens = S.sum(1).astype(np.float64)
    max_len = lens.max()

    unused: dict[int, list[list[str]]] = {}
    used: dict[int, list[list[str]]] = {}
    for pid in sorted(int(p) for p in df["participant_id"].unique()):
        sub = df[df["participant_id"] == pid]
        pool = [[] for _ in range(G)]
        for uid, g in zip(sub["uid"], sub["sign"]):
            pool[gidx[g]].append(uid)
        for lst in pool:
            rng.shuffle(lst)
        unused[pid] = pool
        used[pid] = [[] for _ in range(G)]
    avail = {p: np.array([len(x) for x in pool]) for p, pool in unused.items()}
    has = {p: a > 0 for p, a in avail.items()}
    eligible = {p: (S > 0).astype(np.int32) @ (~h).astype(np.int32) == 0 for p, h in has.items()}
    uses = {p: np.zeros(N) for p in unused}

    plan, orphans = [], 0
    total = remaining = len(df)
    while remaining > 0:
        p = max(avail, key=lambda q: avail[q].sum())
        a = avail[p]
        fresh = np.minimum(S, a[None, :]).sum(1)
        score = (fresh - reuse_penalty * (lens - fresh)) / lens - repeat_penalty * uses[p]
        score += length_bonus * fresh / max_len
        if pressure_weight:
            pressure = (S @ a) / lens / max(a[a > 0].mean(), 1e-9)
            score += pressure_weight * pressure
        score += rng.random(N) * 1e-6
        score[~eligible[p] | (fresh == 0)] = -np.inf
        if not np.isfinite(score).any():
            # this signer's leftover clips are of glosses no eligible sentence
            # contains -- they cannot be placed in a sentence at all
            orphans += int(a.sum())
            remaining -= int(a.sum())
            avail[p] = np.zeros_like(a)
            continue
        k = int(np.argmax(score))
        uses[p][k] += 1
        uids, reused = [], []
        for g in sentences[k].glosses:
            j = gidx[g]
            if unused[p][j]:
                uid = unused[p][j].pop()
                used[p][j].append(uid)
                a[j] -= 1
                remaining -= 1
                uids.append(uid)
                reused.append(False)
            else:
                uids.append(used[p][j][int(rng.integers(len(used[p][j])))])
                reused.append(True)
        plan.append({
            "kind": "sentence",
            "sentence_id": sentences[k].id,
            "participant_id": int(p),
            "glosses": list(sentences[k].glosses),
            "source_uids": uids,
            "reused": reused,
            **_synth_lengths(len(uids), rng, gap_range, rest_range),
        })
        if progress is not None:
            progress(total - remaining, total)

    n_slots = sum(len(r["source_uids"]) for r in plan)
    n_reused = sum(sum(r["reused"]) for r in plan)
    stats = {
        "n_sequences": len(plan),
        "n_slots": n_slots,
        "n_reused_slots": n_reused,
        "reuse_rate": n_reused / max(n_slots, 1),
        "n_clips": total,
        "n_orphan_clips": orphans,
        "n_distinct_sentences": len({r["sentence_id"] for r in plan}),
    }
    return plan, stats


def plan_control(
    df: pd.DataFrame,
    lengths: np.ndarray,
    rng: np.random.Generator,
    *,
    gap_range: tuple[int, int],
    rest_range: tuple[int, int],
) -> list[dict]:
    """Every clip exactly once, random gloss order, one signer per sequence;
    sequence lengths drawn from ``lengths`` (pass the sentence split's
    lengths so the two splits differ only in word order/meaning)."""
    plan = []
    for pid in sorted(int(p) for p in df["participant_id"].unique()):
        sub = df[df["participant_id"] == pid]
        idx = rng.permutation(len(sub))
        uids = sub["uid"].to_numpy()[idx]
        signs = sub["sign"].to_numpy()[idx]
        i = 0
        while i < len(uids):
            n = int(rng.choice(lengths))
            if len(uids) - (i + n) == 1:  # never leave a 1-sign tail
                n += 1
            j = min(i + n, len(uids))
            plan.append({
                "kind": "control",
                "sentence_id": "",
                "participant_id": int(pid),
                "glosses": [str(s) for s in signs[i:j]],
                "source_uids": [str(u) for u in uids[i:j]],
                "reused": [False] * (j - i),
                **_synth_lengths(j - i, rng, gap_range, rest_range),
            })
            i = j
    order = rng.permutation(len(plan))  # interleave signers
    return [plan[i] for i in order]


# ---------------------------------------------------------------------------
# materialize
# ---------------------------------------------------------------------------

def load_clip(data_dir: Path, relpath: str) -> np.ndarray:
    """One GISLR_Stratified clip in canonical row order, NaN preserved."""
    return load_npz(Path(data_dir) / relpath, _ALL_ROWS, "xyz")


def transition(a: np.ndarray, b: np.ndarray, n: int) -> np.ndarray:
    """``n`` frames strictly between frame ``a`` and frame ``b`` (endpoints
    excluded); NaN at either end propagates."""
    t = (np.arange(1, n + 1, dtype=np.float32) / (n + 1))[:, None, None]
    return a[None] + t * (b - a)[None]


def rest(edge: np.ndarray, n: int, rng: np.random.Generator, jitter: float) -> np.ndarray:
    """``n`` frames of ``edge`` held still with Gaussian jitter, hands NaN."""
    out = np.repeat(edge[None], n, axis=0).astype(np.float32)
    out += rng.normal(0.0, jitter, size=out.shape).astype(np.float32)
    out[:, HAND_ROWS, :] = np.nan
    return out


def materialize(
    row: dict,
    data_dir: Path,
    relpath_of: dict[str, str],
    label_of: dict[str, int],
    rng: np.random.Generator,
    *,
    rest_jitter: float,
) -> dict:
    """One plan row -> ``{landmarks, frame_labels, frame_kind, segments, labels}``.

    ``segments[i] = (start, end)``, end exclusive, in output frames: slicing
    ``landmarks[start:end]`` returns source clip ``i`` bit-for-bit.
    """
    clips = [load_clip(data_dir, relpath_of[u]) for u in row["source_uids"]]
    labels = [label_of[g] for g in row["glosses"]]
    parts, kinds, flabels, segments = [], [], [], []
    t = 0

    def add(arr, kind, label):
        nonlocal t
        parts.append(arr)
        kinds.append(np.full(len(arr), kind, np.uint8))
        flabels.append(np.full(len(arr), label, np.int16))
        t += len(arr)

    add(rest(clips[0][0], row["rest_in"], rng, rest_jitter), REST, NULL_LABEL)
    for i, (clip, lab) in enumerate(zip(clips, labels)):
        if i > 0:
            add(transition(clips[i - 1][-1], clip[0], row["gaps"][i - 1]), TRANSITION, NULL_LABEL)
        segments.append((t, t + len(clip)))
        add(clip, SIGN, lab)
    add(rest(clips[-1][-1], row["rest_out"], rng, rest_jitter), REST, NULL_LABEL)

    landmarks = np.concatenate(parts).astype(np.float32)
    validate_tensor(landmarks, dtype=np.float32, check_values=True, where="sequence")
    return {
        "landmarks": landmarks,
        "frame_labels": np.concatenate(flabels),
        "frame_kind": np.concatenate(kinds),
        "segments": np.asarray(segments, dtype=np.int32).reshape(-1, 2),
        "labels": np.asarray(labels, dtype=np.int16),
    }


def write_sequence(path: Path, arrays: dict, dtype=np.float32) -> Path:
    """Atomic compressed npz (temp file + ``os.replace``). ``dtype`` is the
    stored landmark precision: float32 (default) keeps every segment
    bit-identical to its float32 source; float16 halves the size."""
    arrays = {**arrays, "landmarks": arrays["landmarks"].astype(dtype)}
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    tmp = path.with_suffix(".tmp.npz")
    np.savez_compressed(tmp, **arrays)
    os.replace(tmp, path)
    return path


def read_sequence(path: Path) -> dict:
    with np.load(path) as d:
        return {k: d[k] for k in d.files}


def check_roundtrip(arrays: dict, row: dict, data_dir: Path, relpath_of: dict[str, str]) -> None:
    """Every segment slices back to its exact source clip; labels and kinds
    agree with the segments. Raises on the first mismatch.

    The source is cast to the stored dtype before comparing, so a float16
    build is checked for exactly what float16 storage promises (the same
    rounding of the same clip), and a float32 build stays bit-exact."""
    fk, fl = arrays["frame_kind"], arrays["frame_labels"]
    covered = np.zeros(len(fk), bool)
    for (s, e), uid, lab in zip(arrays["segments"], row["source_uids"], arrays["labels"]):
        src = load_clip(data_dir, relpath_of[uid]).astype(arrays["landmarks"].dtype)
        if not np.array_equal(arrays["landmarks"][s:e], src, equal_nan=True):
            raise AssertionError(f"segment {s}:{e} != source clip {uid}")
        if not ((fk[s:e] == SIGN).all() and (fl[s:e] == lab).all()):
            raise AssertionError(f"labels/kinds wrong inside segment {s}:{e}")
        covered[s:e] = True
    if (fk[~covered] == SIGN).any() or (fl[~covered] != NULL_LABEL).any():
        raise AssertionError("a frame outside every segment is labelled as a sign")


# ---------------------------------------------------------------------------
# gap corpus (TODO §17): a boundary/gap-detector needs many transition
# examples, not linguistic sentence structure, and the realistic-rest fix
# `sb.recognize.continuous.data` already found (rest at signing height makes
# "hands NaN" a trivial gap tell -- TODO §12.1's last item). This section
# ports that fix into canonical (T, 543, 3) row space (continuous.data works
# in flattened (subset, coords) feature space instead) and adds a
# coverage-free, many-pass planner: a gap detector does not need every clip
# placed exactly once, so the same clip can recur in different transition
# contexts across passes -- more, more varied examples, not a flaw.
# ---------------------------------------------------------------------------

_POSE0 = next(g.offset for g in GROUPS if g.name == "pose")
_LH_WRIST, _RH_WRIST = 468, 522  # hand landmark 0 of each hand block
_POSE_WRIST = (_POSE0 + 15, _POSE0 + 16)  # left, right
_POSE_ELBOW = (_POSE0 + 13, _POSE0 + 14)
_POSE_HIP = (_POSE0 + 23, _POSE0 + 24)
_LEFT_HAND_ROWS = np.arange(468, 489)
_RIGHT_HAND_ROWS = np.arange(522, 543)


def _hip_y(frame: np.ndarray) -> float | None:
    hips = frame[list(_POSE_HIP), 1]
    good = hips[np.isfinite(hips)]
    return float(good.mean()) if good.size else None


def _lowered_frame(frame: np.ndarray, default_drop: float) -> np.ndarray:
    """``frame`` (543, 3) with both hands and the pose wrists/elbows moved
    down to hip height, y only, NaN preserved -- the canonical-row-space twin
    of ``sb.recognize.continuous.data._lowered``."""
    out = frame.copy()
    hip = _hip_y(frame)

    def drop(wrist_row: int) -> float:
        y = frame[wrist_row, 1]
        return max(hip - y, 0.05) if hip is not None and np.isfinite(y) else default_drop

    for hand_rows, wrist_row in ((_LEFT_HAND_ROWS, _LH_WRIST), (_RIGHT_HAND_ROWS, _RH_WRIST)):
        out[hand_rows, 1] += drop(wrist_row)
    for pose_wrist, pose_elbow in zip(_POSE_WRIST, _POSE_ELBOW):
        d = drop(pose_wrist)
        out[pose_wrist, 1] += d
        out[pose_elbow, 1] += d / 2
    return out


def lowered_rest(edge: np.ndarray, n: int, entering: bool, rng: np.random.Generator, *,
                 ramp_frames: int, hand_visible_max: float, default_drop: float, jitter: float) -> np.ndarray:
    """``n`` realistic rest frames next to ``edge`` (543, 3): the arms ramp
    between ``edge`` and the lowered pose over the first/last ``ramp_frames``
    (toward the sign when ``entering``), and each hand goes NaN on every
    frame where its wrist has dropped below ``hand_visible_max`` (out of
    frame) -- fixes the flaw in :func:`rest` (hands always NaN, pose left at
    signing height), where "hands NaN" alone gives away the gap."""
    low = _lowered_frame(edge, default_drop)
    k = min(ramp_frames, n)
    t = np.clip(np.arange(1, n + 1) / (k + 1), 0, 1)[:, None, None]
    frames = edge[None] + t * (low - edge)[None]  # leaving: edge -> low
    if entering:
        frames = frames[::-1]  # arriving: low -> edge
    for hand_rows, wrist_row in ((_LEFT_HAND_ROWS, _LH_WRIST), (_RIGHT_HAND_ROWS, _RH_WRIST)):
        gone = np.flatnonzero(~(frames[:, wrist_row, 1] <= hand_visible_max))
        if gone.size:
            frames[np.ix_(gone, hand_rows)] = np.nan
    frames = frames + rng.normal(0.0, jitter, size=frames.shape).astype(np.float32)
    return frames.astype(np.float32)


def materialize_v2(
    row: dict,
    data_dir: Path,
    relpath_of: dict[str, str],
    label_of: dict[str, int],
    rng: np.random.Generator,
    *,
    rest_jitter: float,
    rest_ramp_frames: int,
    hand_visible_max: float,
    rest_default_drop: float,
    p_lowered_rest: float,
) -> dict:
    """Like :func:`materialize`, but each sequence's rest is **lowered**
    (realistic) with probability ``p_lowered_rest`` and hands-absent
    (:func:`rest`, GISLR-Sentences v1's style) otherwise -- kept as a mix so
    a model trained on this also handles the v1 style. Adds ``gap`` (uint8,
    1 on every non-sign frame): the label a boundary/gap detector trains on,
    equivalent to ``frame_kind != SIGN`` but named for that reader."""
    clips = [load_clip(data_dir, relpath_of[u]) for u in row["source_uids"]]
    labels = [label_of[g] for g in row["glosses"]]
    parts, kinds, flabels, segments = [], [], [], []
    t = 0
    lowered = rng.random() < p_lowered_rest

    def add(arr, kind, label):
        nonlocal t
        parts.append(arr)
        kinds.append(np.full(len(arr), kind, np.uint8))
        flabels.append(np.full(len(arr), label, np.int16))
        t += len(arr)

    def make_rest(edge, n, entering):
        if lowered:
            return lowered_rest(edge, n, entering, rng, ramp_frames=rest_ramp_frames,
                                hand_visible_max=hand_visible_max, default_drop=rest_default_drop,
                                jitter=rest_jitter)
        return rest(edge, n, rng, rest_jitter)

    add(make_rest(clips[0][0], row["rest_in"], True), REST, NULL_LABEL)
    for i, (clip, lab) in enumerate(zip(clips, labels)):
        if i > 0:
            add(transition(clips[i - 1][-1], clip[0], row["gaps"][i - 1]), TRANSITION, NULL_LABEL)
        segments.append((t, t + len(clip)))
        add(clip, SIGN, lab)
    add(make_rest(clips[-1][-1], row["rest_out"], False), REST, NULL_LABEL)

    landmarks = np.concatenate(parts).astype(np.float32)
    validate_tensor(landmarks, dtype=np.float32, check_values=True, where="sequence")
    frame_kind = np.concatenate(kinds)
    return {
        "landmarks": landmarks,
        "frame_labels": np.concatenate(flabels),
        "frame_kind": frame_kind,
        "gap": (frame_kind != SIGN).astype(np.uint8),
        "segments": np.asarray(segments, dtype=np.int32).reshape(-1, 2),
        "labels": np.asarray(labels, dtype=np.int16),
        "lowered_rest": bool(lowered),
    }


def plan_control_multi(df: pd.DataFrame, lengths: np.ndarray, seed: int, *,
                       gap_range: tuple[int, int], rest_range: tuple[int, int], n_passes: int) -> list[dict]:
    """:func:`plan_control` run ``n_passes`` times, each with its own
    ``[seed, pass]``-derived RNG, tagged ``pass_id``, and concatenated.

    A gap/boundary detector's signal is the transition itself, not sentence
    semantics or exactly-once clip coverage, so re-drawing the same clip pool
    into different random neighbours across passes is a deliberate way to
    grow and diversify a "very big" corpus, not a coverage bug (unlike
    :func:`plan_sentences`, which places every clip exactly once by design)."""
    out = []
    for p in range(n_passes):
        prng = np.random.default_rng([seed, p])
        rows = plan_control(df, lengths, prng, gap_range=gap_range, rest_range=rest_range)
        for r in rows:
            r["pass_id"] = p
        out.extend(rows)
    return out


def summarize_row_v2(row: dict, arrays: dict, seq_id: str, relpath: str) -> dict:
    """:func:`summarize_row` plus the fields :func:`plan_control_multi` /
    :func:`materialize_v2` add: ``pass_id`` (which of the ``n_passes`` draws
    placed this clip pool) and ``lowered_rest`` (which rest style this
    sequence got)."""
    return {**summarize_row(row, arrays, seq_id, relpath),
            "pass_id": row.get("pass_id", 0), "lowered_rest": bool(arrays.get("lowered_rest", False))}


def summarize_row(row: dict, arrays: dict, seq_id: str, relpath: str) -> dict:
    """The ``sequences.csv`` row for one materialized sequence."""
    seg = arrays["segments"]
    return {
        "seq_id": seq_id,
        "kind": row["kind"],
        "sentence_id": row["sentence_id"],
        "participant_id": row["participant_id"],
        "n_signs": len(row["glosses"]),
        "glosses": " ".join(row["glosses"]),
        "labels": " ".join(str(int(x)) for x in arrays["labels"]),
        "starts": " ".join(str(int(s)) for s in seg[:, 0]),
        "ends": " ".join(str(int(e)) for e in seg[:, 1]),
        "source_uids": " ".join(row["source_uids"]),
        "reused": " ".join(str(int(r)) for r in row["reused"]),
        "gaps": " ".join(str(g) for g in row["gaps"]),
        "rest_in": row["rest_in"],
        "rest_out": row["rest_out"],
        "n_frames": int(len(arrays["frame_kind"])),
        "npz_relpath": relpath,
    }
'''


def _register(name: str, source: str) -> types.ModuleType:
    parts = name.split(".")
    for i in range(1, len(parts)):
        parent = ".".join(parts[:i])
        if parent not in sys.modules:
            pkg = types.ModuleType(parent)
            pkg.__path__ = []
            sys.modules[parent] = pkg
    module = types.ModuleType(name)
    module.__file__ = f"<embedded {name}>"
    sys.modules[name] = module
    exec(compile(source, module.__file__, "exec"), module.__dict__)
    setattr(sys.modules[".".join(parts[:-1])], parts[-1], module)
    return module


for _name, _src in _SOURCES.items():
    _register(_name, _src)
assert hashlib.sha256("".join(_SOURCES.values()).encode()).hexdigest() == CODE_SHA256
compose = sys.modules["sb.recognize.sequences.compose"]
print(f"modules: {list(_SOURCES)}")

## 1. Label map + source pool

Both `train.csv` and `test.csv` combined into one clip pool (a gap detector's task
does not depend on the 250-gloss classification split those files encode). Signers
are split into this corpus's own train/test groups, disjoint by participant.

In [ ]:
# ============================================================
# Label map + combined clip pool + signer-disjoint split
# ============================================================
train_df = pd.read_csv(DATA_DIR / "train.csv")
test_df = pd.read_csv(DATA_DIR / "test.csv")
signs = set(train_df["sign"]) | set(test_df["sign"])
LABEL_MAP = {sign: i for i, sign in enumerate(sorted(signs))}
(OUT_DIR / "sign_to_prediction_index_map.json").write_text(json.dumps(LABEL_MAP), encoding="utf-8")

pool = pd.concat([train_df.assign(source_split="train"), test_df.assign(source_split="test")],
                 ignore_index=True)
srng = np.random.default_rng(CONFIG["seed"])
participants = np.sort(pool["participant_id"].unique())
srng.shuffle(participants)
n_train_signers = int(round(len(participants) * CONFIG["train_signer_frac"]))
signer_split = {int(p): ("train" if i < n_train_signers else "test") for i, p in enumerate(participants)}
pool["split"] = pool["participant_id"].map(signer_split)
relpath_of = dict(zip(pool["uid"], pool["npz_relpath"]))

print(f"pool: {len(pool)} clips ({len(train_df)} train.csv + {len(test_df)} test.csv), "
      f"{len(signs)} glosses, {len(participants)} signers "
      f"({n_train_signers} train / {len(participants) - n_train_signers} test)")
print(pool["split"].value_counts())

## 2. Plan: many random passes over the whole pool

`plan_control_multi` -- no sentence structure, no coverage requirement: each of
`n_passes` draws places every clip once, in a fresh random order and neighbour set.

In [ ]:
# ============================================================
# Plan (resumable in `plan.json`)
# ============================================================
if PLAN_PATH.exists() and json.loads(PLAN_PATH.read_text())["config_sha256"] == CONFIG_SHA:
    PLAN = json.loads(PLAN_PATH.read_text())
    print("plan loaded")
else:
    lengths = np.arange(CONFIG["signs_per_seq"][0], CONFIG["signs_per_seq"][1] + 1)
    rows = compose.plan_control_multi(
        pool, lengths, CONFIG["seed"],
        gap_range=tuple(CONFIG["gap_frames"]), rest_range=tuple(CONFIG["rest_frames"]),
        n_passes=CONFIG["n_passes"])
    for i, r in enumerate(rows):
        r["seq_id"] = f"g{i:06d}"
        r["split"] = signer_split[r["participant_id"]]
    PLAN = {"config_sha256": CONFIG_SHA, "sequences": rows}
    write_json(PLAN_PATH, PLAN)

by_split = pd.Series([r["split"] for r in PLAN["sequences"]]).value_counts()
print(f"{len(PLAN['sequences'])} sequences over {CONFIG['n_passes']} passes")
print(by_split)

## 3. Materialize the sequences

Manifest-driven and resumable, like GISLR-Sentences. Rest is lowered (realistic) or
hands-absent per sequence, drawn once per row so `lowered_rest` in the manifest
matches what's actually in the npz. Size is projected after the first 300 sequences.

In [ ]:
# ============================================================
# Build every sequence npz
# ============================================================
PROJECT_AFTER = 300

rows = PLAN["sequences"]
dtype = np.dtype(STORAGE_DTYPE)
manifest = json.loads(MANIFEST_PATH.read_text()) if MANIFEST_PATH.exists() else {}


def build(i_row):
    i, row = i_row
    rel = f"sequences/{row['seq_id']}.npz"
    try:
        arrays = compose.materialize_v2(
            row, DATA_DIR, relpath_of, LABEL_MAP, np.random.default_rng([CONFIG["seed"], i]),
            rest_jitter=CONFIG["rest_jitter"], rest_ramp_frames=CONFIG["rest_ramp_frames"],
            hand_visible_max=CONFIG["hand_visible_max"], rest_default_drop=CONFIG["rest_default_drop"],
            p_lowered_rest=CONFIG["p_lowered_rest"])
        compose.write_sequence(OUT_DIR / rel, arrays, dtype=dtype)
        rec = compose.summarize_row_v2(row, arrays, row["seq_id"], rel)
        return row["seq_id"], {"status": "done", "bytes": (OUT_DIR / rel).stat().st_size,
                                "split": row["split"], **rec}
    except Exception as e:
        return row["seq_id"], {"status": "failed", "error": repr(e)}


todo = [(i, r) for i, r in enumerate(rows) if manifest.get(r["seq_id"], {}).get("status") != "done"]
bar = tqdm(total=len(rows), initial=len(rows) - len(todo), desc="materialize")
n_failed = 0
with ThreadPoolExecutor(N_WORKERS) as ex:
    for k, (seq_id, rec) in enumerate(ex.map(build, todo), 1):
        manifest[seq_id] = rec
        n_failed += rec["status"] == "failed"
        bar.update(1)
        done_bytes = [v["bytes"] for v in manifest.values() if v["status"] == "done"]
        projected = np.mean(done_bytes) * len(rows) / 1e9 if done_bytes else 0.0
        bar.set_postfix(failed=n_failed, projected_gb=f"{projected:.1f}")
        if k == PROJECT_AFTER and projected > OUTPUT_LIMIT_GB:
            write_json(MANIFEST_PATH, manifest)
            raise RuntimeError(
                f"projected {projected:.1f} GB > OUTPUT_LIMIT_GB={OUTPUT_LIMIT_GB}. Lower n_passes in "
                f"the config, or set STORAGE_DTYPE = 'float16'; delete {SEQ_DIR} and {MANIFEST_PATH}, "
                f"and re-run.")
        if k % 1000 == 0:
            write_json(MANIFEST_PATH, manifest)
write_json(MANIFEST_PATH, manifest)
bar.close()
print(f"{sum(v['status'] == 'done' for v in manifest.values())}/{len(rows)} done, {n_failed} failed this run")

## 4. train.csv / test.csv, build record

In [ ]:
# ============================================================
# train.csv / test.csv (gloss list + segment frames), build_info.json
# ============================================================
manifest = json.loads(MANIFEST_PATH.read_text())
failed = [k for k, v in manifest.items() if v["status"] != "done"]
assert not failed, f"{len(failed)} sequences not done -- re-run section 3: {failed[:5]}"

order = [r["seq_id"] for r in PLAN["sequences"]]
full_df = pd.DataFrame([{k: v for k, v in manifest[s].items() if k not in ("status", "bytes")} for s in order])
for split in ("train", "test"):
    full_df[full_df["split"] == split].drop(columns=["split"]).to_csv(OUT_DIR / f"{split}.csv", index=False)

frames = full_df.groupby("split")["n_frames"].agg(["count", "sum", "mean", "max"])
gap_rate = {}
for seq_id in full_df["seq_id"].sample(min(500, len(full_df)), random_state=0):
    arr = compose.read_sequence(OUT_DIR / full_df.set_index("seq_id").loc[seq_id, "npz_relpath"])
    gap_rate[seq_id] = float(arr["gap"].mean())
size_bytes = sum(p.stat().st_size for p in SEQ_DIR.rglob("*.npz"))
build_info = {
    "dataset": "GISLR-GapCorpus", "dataset_version": CONFIG["dataset_version"],
    "built_on": "kaggle", "storage_dtype": STORAGE_DTYPE,
    "config": CONFIG, "config_sha256": CONFIG_SHA, "code_sha256": CODE_SHA256,
    "signbridge_commit": SIGNBRIDGE_COMMIT,
    "source": {"kaggle_ref": "bracu23101281/gislr-stratified", "resolved_dir": str(DATA_DIR)},
    "frames": frames.reset_index().to_dict(orient="records"),
    "mean_gap_fraction_sampled": float(np.mean(list(gap_rate.values()))),
    "size_bytes": size_bytes,
}
write_json(OUT_DIR / "build_info.json", build_info)

card = f"""# GISLR-GapCorpus {CONFIG['dataset_version']}

Sign-vs-gap training corpus (signbridge TODO §17): random-order, random-length landmark
sequences from **every** GISLR_Stratified clip (`train.csv` + `test.csv`), drawn over
{CONFIG['n_passes']} independent passes -- no sentence structure, no exactly-once coverage
requirement, since the target is the transition itself, not language.

## Files
- `sequences/<seq_id>.npz`: `landmarks (T,543,3) {STORAGE_DTYPE}` (canonical gislr-holistic
  row order), `frame_labels (T,) int16` (-1 null), `frame_kind (T,) uint8` (0 sign /
  1 transition / 2 rest), **`gap (T,) uint8`** (1 on every non-sign frame -- the
  training target), `segments (n,2) int32`, `labels (n,) int16`, `lowered_rest` (bool).
- `train.csv` / `test.csv`: one row per sequence -- `glosses`, `labels`, `starts`,
  `ends` (frame indices; a sign occupies `[start, end)`, everything else is a gap),
  `n_frames`, `npz_relpath`, `pass_id`, `lowered_rest`, `participant_id`.
  Split by **signer** ({CONFIG['train_signer_frac']:.0%} of participants in train), independent
  of GISLR_Stratified's own clip split.
- `sign_to_prediction_index_map.json`, `plan.json`, `build_info.json`.

## Rest synthesis
- **lowered** (probability {CONFIG['p_lowered_rest']}): arms ramp to hip height over
  {CONFIG['rest_ramp_frames']} frames; each hand goes NaN only once its wrist crosses
  {CONFIG['hand_visible_max']} (out of frame) -- realistic, fixes GISLR-Sentences v1's
  "hands NaN always means gap" tell.
- **hands-absent** (otherwise): edge frame held, both hands NaN throughout -- kept in
  the mix so a model trained here also handles that simpler case.

## Caveats
- Non-sign frames are synthetic (interpolated transitions, synthesized rest); real
  continuous signing was not used to build this. Treat any number as an upper bound.
- {CONFIG['n_passes']}x reuse of the same clip pool means the *same* clip appears in
  several different neighbour contexts across passes -- deliberate for a boundary
  task, but this is not a source of {CONFIG['n_passes']}x more distinct hand shapes.
"""
(OUT_DIR / "README.md").write_text(card, encoding="utf-8")
print(frames)
print(f"mean gap fraction (500-sequence sample): {build_info['mean_gap_fraction_sampled']:.3f}")
print(f"size: {size_bytes / 1e9:.2f} GB ({STORAGE_DTYPE})")

## 5. Checks and figures

In [ ]:
# ============================================================
# Round-trip sample + gap-label consistency
# ============================================================
by_id = {r["seq_id"]: r for r in PLAN["sequences"]}
rng = np.random.default_rng(CONFIG["seed"])
sample = sorted(rng.choice(full_df["seq_id"].to_numpy(), CONFIG["n_roundtrip_checks"], replace=False).tolist())
bad = []
for seq_id in tqdm(sample, desc="round-trip"):
    arrays = compose.read_sequence(OUT_DIR / full_df.set_index("seq_id").loc[seq_id, "npz_relpath"])
    try:
        compose.check_roundtrip(arrays, by_id[seq_id], DATA_DIR, relpath_of)
        assert np.array_equal(arrays["gap"], (arrays["frame_kind"] != compose.SIGN).astype(np.uint8))
    except AssertionError as e:
        bad.append((seq_id, str(e)))
write_json(OUT_DIR / "roundtrip_sample.json", {"seed": CONFIG["seed"], "seq_ids": sample, "failures": bad})
print(f"round-trip: {len(sample) - len(bad)}/{len(sample)} exact at {STORAGE_DTYPE}")
assert not bad, "checks failed -- do not publish this output"

In [ ]:
# ============================================================
# Figures: lengths + one exemplar sequence (both rest styles)
# ============================================================
fig, axes = plt.subplots(1, 2, figsize=(11, 3.8))
axes[0].hist(full_df["n_frames"], bins=60, color="#4c72b0")
axes[0].set_title(f"{len(full_df)} sequences, mean {full_df.n_frames.mean():.0f} frames")
axes[0].set_xlabel("frames")
full_df["lowered_rest"].value_counts().plot.bar(ax=axes[1], color=["#dd8452", "#4c72b0"])
axes[1].set_title("rest style")
fig.tight_layout()
fig.savefig(ASSETS_DIR / "lengths.png", dpi=110)
plt.show()

for style, want_lowered in (("lowered", True), ("hands-absent", False)):
    ex = full_df[full_df["lowered_rest"] == want_lowered].iloc[0]
    arr = compose.read_sequence(OUT_DIR / ex["npz_relpath"])
    fig, ax = plt.subplots(figsize=(14, 3.2))
    for idx, name in ((489 + 15, "left wrist"), (489 + 16, "right wrist")):
        ax.plot(-arr["landmarks"][:, idx, 1], label=f"pose {name} (-y)")
    colors = {compose.TRANSITION: "#dddddd", compose.REST: "#f4cccc"}
    for t, k in enumerate(arr["frame_kind"]):
        if k in colors:
            ax.axvspan(t - 0.5, t + 0.5, color=colors[k], lw=0)
    for (s, e), gl in zip(arr["segments"], ex["glosses"].split()):
        ax.text((s + e) / 2, ax.get_ylim()[1], gl.upper(), ha="center", va="bottom")
    ax.set_title(f"{ex['seq_id']} ({style} rest): grey = transition, pink = rest")
    ax.legend(loc="lower right")
    fig.tight_layout()
    fig.savefig(ASSETS_DIR / f"exemplar_{style}.png", dpi=110)
    plt.show()

## 6. Clean up, then publish

Build scratch removed so the output folder is exactly the dataset. From the committed
version's **Output** tab: **New Dataset** → e.g. `GISLR-GapCorpus`. Private by default.

In [ ]:
# ============================================================
# Remove build scratch from the output folder
# ============================================================
MANIFEST_PATH.unlink(missing_ok=True)
for p in OUT_DIR.rglob("*.tmp*"):
    p.unlink()
n_files = sum(1 for p in OUT_DIR.rglob("*") if p.is_file())
total = sum(p.stat().st_size for p in OUT_DIR.rglob("*") if p.is_file())
print(f"{OUT_DIR}: {n_files} files, {total / 1e9:.2f} GB -- ready to publish as a dataset")